# GRPO: Reinforcement Learning for DSPy Programs (Chapter 6)

This notebook accompanies Chapter 6 §6.8 of *Context Engineering with DSPy*.

> **Requires multiple CUDA GPUs AND a running [Arbor](https://github.com/Ziems/arbor) server.** GRPO is the most expensive optimizer in DSPy because it runs an RL loop on a local model. The notebook initializes Arbor in-process; you need at least 2 CUDA GPUs (one for inference via vLLM, one for training via TRL). Without a GPU it will error clearly on the first cell that tries to start the Arbor server.

For a smoke test, lower `num_train_steps` to 10 in the GRPO config cell before running.


## Setup

GRPO requires both DSPy and the Arbor RL framework. Arbor handles the complexity of splitting GPUs between inference (vLLM) and training (TRL/GRPO).

In [ ]:
# Core dependencies
%pip install -r ../requirements.txt -q

# Arbor RL framework (handles GPU orchestration for GRPO)

# Latest DSPy from main (GRPO support may require latest)
# !pip install -U git+https://github.com/stanfordnlp/dspy.git@main -q

# Optional: flash-attn for faster inference
# !pip install flash-attn --no-build-isolation -q


In [ ]:
import dspy
import torch
import random
import os
import time
import logging
from dotenv import load_dotenv

load_dotenv()
logging.basicConfig(level=logging.INFO)

print(f"DSPy version: {dspy.__version__}")
print(f"PyTorch version: {torch.__version__}")

In [ ]:
# Verify CUDA GPUs are available
if not torch.cuda.is_available():
    raise RuntimeError(
        "GRPO requires NVIDIA GPUs with CUDA. "
        "This notebook cannot run on Mac (MPS) or CPU-only machines."
    )

num_gpus = torch.cuda.device_count()
print(f"Available GPUs: {num_gpus}")
for i in range(num_gpus):
    name = torch.cuda.get_device_name(i)
    vram = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f"  GPU {i}: {name} ({vram:.1f} GB VRAM)")

if num_gpus < 2:
    print("\n⚠️ GRPO needs at least 2 GPUs (1 inference + 1 training).")
    print("You can still read through the notebook, but compile() will fail.")

## Initialize Arbor

Arbor is the RL infrastructure that powers `dspy.GRPO`. When you call `arbor.init()`, it starts a background server that manages GPU allocation — splitting your GPUs between a vLLM inference server (for running rollouts) and TRL training workers (for updating model weights).

The `ArborProvider` is a DSPy `Provider` subclass that connects DSPy's LM interface to the Arbor server. Models are prefixed with `openai/arbor:` so DSPy routes requests through Arbor instead of a regular API.

In [ ]:
import arbor
from dspy.clients.lm_local_arbor import ArborProvider

# Start the Arbor server (manages GPU allocation in the background)
arbor_server_info = arbor.init()
print(f"Arbor server running at: {arbor_server_info['api_base']}")

In [ ]:
# Configure the local model via Arbor
LOCAL_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"  # Small but capable

local_lm = dspy.LM(
    model=f"openai/arbor:{LOCAL_MODEL}",
    provider=ArborProvider(),
    temperature=0.7,
    api_base=arbor_server_info["api_base"],
)

dspy.configure(lm=local_lm)
print(f"Model: {LOCAL_MODEL}")
print(f"Provider: ArborProvider (inference via vLLM, training via TRL)")

## Load Dataset

Same AI vs Human text detection task from the optimizer benchmark.

In [ ]:
import pandas as pd

csv_path = '../data/ai_vs_human200.csv'
df = pd.read_csv(csv_path)
examples = df.to_dict(orient='records')

dataset = [
    dspy.Example(**ex).with_inputs("text")
    for ex in examples
]

random.seed(42)
random.shuffle(dataset)

n = len(dataset)
train_end = int(n * 0.5)
val_end = int(n * 0.75)

trainset = dataset[:train_end]
valset = dataset[train_end:val_end]
testset = dataset[val_end:]

print(f"Training:   {len(trainset)} examples")
print(f"Validation: {len(valset)} examples")
print(f"Test:       {len(testset)} examples")

## Define Module and Metric

In [ ]:
class DetectAIText(dspy.Signature):
    """Analyze text and determine if it was written by an AI or a human."""
    text: str = dspy.InputField(description="The text to analyze")
    is_ai: bool = dspy.OutputField(description="True if AI-generated, False if human-written")


class AIDetector(dspy.Module):
    def __init__(self):
        super().__init__()
        self.detect = dspy.ChainOfThought(DetectAIText)

    def forward(self, text: str):
        return self.detect(text=text)


def exact_match(example, prediction, trace=None):
    """Reward function: 1 for correct, 0 for incorrect."""
    return 1 if example.is_ai == prediction.is_ai else 0


print("Module: AIDetector (ChainOfThought)")
print("Metric: exact_match (used as reward signal for GRPO)")

## Baseline Evaluation

Before running GRPO, let's see how the base model performs without any optimization.

In [ ]:
program = AIDetector()
program.set_lm(local_lm)

evaluator = dspy.Evaluate(
    devset=valset,
    metric=exact_match,
    num_threads=16,
    display_progress=True,
    display_table=False,
)

baseline_score = evaluator(program)
print(f"\nBaseline accuracy: {baseline_score:.1f}%")

## Run GRPO Optimization

GRPO works differently from BootstrapFinetune. Instead of learning from a teacher's demonstrations, it:

1. **Rolls out** the current model on training examples multiple times
2. **Scores** each rollout using your metric (the reward signal)
3. **Updates** model weights to make high-reward responses more likely
4. **Repeats** for the configured number of training steps

The key parameters:
- `num_dspy_examples_per_grpo_step`: How many training examples to sample per step
- `num_rollouts_per_grpo_step`: How many times to run each example (more rollouts = better reward estimates)
- `num_train_steps`: Total RL training steps
- `exclude_demos`: Must be `True` — GRPO doesn't use few-shot demos
- `multitask`: Must be `True` — trains all modules together
- `gpu_config`: How to split GPUs between inference and training

In [ ]:
# GPU allocation: 1 for vLLM inference, rest for training.
# The inference/training GPU split is configured on the Arbor server
# (see your Arbor config), not through the GRPO optimizer.
NUM_INFERENCE_GPUS = 1
NUM_TRAINING_GPUS = max(1, num_gpus - NUM_INFERENCE_GPUS)

# RL training hyperparameters
train_kwargs = {
    "per_device_train_batch_size": 2,
    "gradient_accumulation_steps": 4,
    "temperature": 0.7,          # Must match LM config above
    "beta": 0.04,                # KL penalty (higher = stay closer to base model)
    "learning_rate": 1e-6,
    "gradient_checkpointing": True,
    "bf16": True,
    "lr_scheduler_type": "constant_with_warmup",
    "scale_rewards": True,
    "max_grad_norm": 0.5,
    "lora": True,                # LoRA for parameter-efficient training
}

print(f"GPU allocation: {NUM_INFERENCE_GPUS} inference + {NUM_TRAINING_GPUS} training")
print(f"\nTraining config:")
for k, v in train_kwargs.items():
    print(f"  {k}: {v}")

In [ ]:
from dspy.teleprompt.grpo import GRPO

# Set up the student program
student = AIDetector()
student.set_lm(local_lm)

compiler = GRPO(
    metric=exact_match,
    num_dspy_examples_per_grpo_step=6,   # Batch size in DSPy examples
    num_rollouts_per_grpo_step=4,        # Rollouts per example
    exclude_demos=True,                   # Required for GRPO
    multitask=True,                       # Required: train all modules together
    num_train_steps=100,                  # Total RL training steps
    num_threads=16,
    use_train_as_val=False,
    num_steps_for_val=10,                 # Validate every 10 steps
    train_kwargs=train_kwargs,
)

print("Starting GRPO optimization...")
print(f"  Training steps: 100")
print(f"  Batch: 6 examples x 4 rollouts = 24 rollouts per step")
print(f"  Model: {LOCAL_MODEL}")
print()

start_time = time.time()
optimized_program = compiler.compile(
    student=student,
    trainset=trainset,
    valset=valset,
)
elapsed = time.time() - start_time

print(f"\nGRPO optimization complete in {elapsed:.0f}s ({elapsed/3600:.1f} hours)")

## Evaluate GRPO-optimized Model

In [ ]:
grpo_score = evaluator(optimized_program)

print("=" * 55)
print("GRPO Results")
print("=" * 55)
print(f"  Baseline accuracy:  {baseline_score:.1f}%")
print(f"  GRPO accuracy:      {grpo_score:.1f}%")
print(f"  Uplift:             +{grpo_score - baseline_score:.1f}%")
print(f"  Training time:      {elapsed:.0f}s")
print(f"  Model:              {LOCAL_MODEL}")
print(f"  Training steps:     100")
print("=" * 55)

In [ ]:
# Test the optimized program on a few examples
for i, example in enumerate(testset[:3]):
    pred = optimized_program(text=example.text)
    correct = "✅" if pred.is_ai == example.is_ai else "❌"
    print(f"Example {i+1}: {correct}")
    print(f"  Text: {example.text[:80]}...")
    print(f"  Predicted: {pred.is_ai} | Actual: {example.is_ai}")
    print()

## Multi-Module GRPO (mmGRPO)

Our `AIDetector` has a single predictor, but GRPO also supports programs with multiple LM calls. This is called **multi-module GRPO (mmGRPO)**, described in the paper by Ziems et al. (2025).

Standard GRPO only trains single-module programs. mmGRPO generalizes to multi-module DSPy programs by:
- Running full program rollouts (all modules execute end-to-end)
- Grouping LM calls **by module across rollouts** (not by trajectory)
- Handling variable-length and interrupted trajectories

The `multitask=True` parameter enables this. Here's what a multi-module program would look like:

In [ ]:
# Example: multi-module program for multi-hop research
# (from DSPy's rl_multihop tutorial)

class ResearchHop(dspy.Module):
    """Multi-hop research: generate queries and append notes across hops."""
    def __init__(self, num_hops=2):
        self.num_hops = num_hops
        # Two modules — GRPO trains both together
        self.generate_query = dspy.ChainOfThought("claim, key_facts -> followup_search_query")
        self.append_notes = dspy.ChainOfThought("claim, key_facts, search_results -> new_key_facts")

    def forward(self, claim: str):
        key_facts = []
        for hop in range(self.num_hops):
            query = self.generate_query(claim=claim, key_facts=key_facts)
            results = search(query.followup_search_query)  # your retrieval function
            notes = self.append_notes(claim=claim, key_facts=key_facts, search_results=results)
            key_facts.append(notes.new_key_facts)
        return dspy.Prediction(key_facts=key_facts)

# mmGRPO trains generate_query and append_notes together.
# Rewards from the end-to-end metric flow back to both modules.
print("ResearchHop has 2 predictors — mmGRPO trains both from end-to-end rewards.")

## The GEPA → GRPO Pipeline

One practical strategy is to run prompt optimization first, then apply GRPO on top. This is the "BetterTogether" concept using online RL instead of supervised fine-tuning:

1. **GEPA/MIPROv2/SIMBA**: Optimize prompts (instructions + few-shot examples) on an API model
2. **GRPO with Arbor**: Apply RL to fine-tune a small local model using the optimized prompts

Prompt optimization teaches the model *what* to do via better instructions. RL teaches it *how* to do it via weight updates. The two techniques compose.

In [ ]:
# Step 1: Optimize prompts with GEPA (or MIPROv2, SIMBA)
# (This can run on any machine — no GPU required)
api_lm = dspy.LM("openai/gpt-4o-mini", api_key=os.getenv("OPENAI_API_KEY"))
dspy.configure(lm=api_lm)

def gepa_feedback(gold, pred, trace=None, pred_name=None, pred_trace=None):
    score = 1.0 if gold.is_ai == pred.is_ai else 0.0
    if pred_name is not None:
        return dspy.Prediction(
            score=score,
            feedback=f"Predicted is_ai={pred.is_ai}, expected is_ai={gold.is_ai}.",
        )
    return score

prompt_optimized = dspy.GEPA(
    metric=gepa_feedback,
    auto="light",
    reflection_lm=api_lm,
).compile(AIDetector(), trainset=trainset)

# Step 2: Apply GRPO on top with a local model
# (Requires multi-GPU setup with Arbor)
dspy.configure(lm=local_lm)

# Transfer the optimized instructions to the local model program
student = AIDetector()
student.set_lm(local_lm)
for s_pred, t_pred in zip(student.predictors(), prompt_optimized.predictors()):
    s_pred.signature = t_pred.signature  # Copy optimized instructions

# Run GRPO on the prompt-optimized program
compiler = GRPO(
    metric=exact_match,
    num_dspy_examples_per_grpo_step=6,
    num_rollouts_per_grpo_step=4,
    exclude_demos=True,
    multitask=True,
    num_train_steps=100,
    num_threads=16,
    train_kwargs=train_kwargs,
)

final_program = compiler.compile(
    student=student,
    trainset=trainset,
    valset=valset,
)

final_score = evaluator(final_program)
print(f"GEPA + GRPO accuracy: {final_score:.1f}%")

## Results from DSPy Tutorials

Since GRPO requires multi-GPU infrastructure that most readers won't have locally, here are results from DSPy's official tutorials to give you a sense of what to expect:

### Multi-Hop Research (HoVer dataset)
- **Model**: Qwen2.5-7B-Instruct
- **Program**: ResearchHop (2 modules: `generate_query` + `append_notes`)
- **Baseline recall**: 61.8%
- **After GRPO**: 66.2% (+4.4%)
- **Training time**: ~18 hours on 4x H100 GPUs

### Privacy-Preserving Delegation (PAPILLON)
- **Model**: Qwen2.5-7B-Instruct + GPT-4.1-mini as untrusted model
- **Program**: PAPILLON (2 modules: `craft_redacted_request` + `respond_to_query`)
- **Baseline score**: 54.6%
- **After GRPO**: 60.0% (+5.4%)
- **Training time**: ~3 hours on 4x H100 GPUs

### mmGRPO Paper Results (Ziems et al., 2025)
- **+11% accuracy** on average vs post-trained base model
- **+5% accuracy** vs prompt optimization alone
- Tested across classification, multi-hop search, and privacy delegation tasks

## Summary

**GRPO uses reinforcement learning instead of supervised fine-tuning:**
- BootstrapFinetune: learns from teacher traces (SFT) — "do exactly what the teacher did"
- GRPO: learns from reward signals (RL) — "figure out what gets the highest reward"

**Arbor handles the infrastructure complexity:**
- Splits GPUs between vLLM (inference) and TRL (training)
- Manages model serving, checkpointing, and LoRA adapters
- You write DSPy code; Arbor handles the RL plumbing

**When to use GRPO:**
- You need to train a small open-source model (1.5B-7B) for autonomous deployment
- Privacy-sensitive environments where you can't use API models
- High-volume inference where per-request API costs are prohibitive
- You've exhausted prompt optimization gains and need further improvement

**When NOT to use GRPO:**
- Prompt optimization meets your accuracy target (it usually does)
- You don't have access to multiple NVIDIA GPUs
- You're still iterating on your program design
- The cost/quality ratio of prompt optimizers is sufficient for your use case

**GRPO is still experimental.** Both DSPy and Arbor mark this as proof-of-concept. For most production use cases, prompt optimization with GEPA or MIPROv2 gives you a better cost/quality tradeoff. But when you need a self-contained, privacy-preserving, high-throughput system, GRPO is the path to get there.